# DSCI 552 Homework 3
Ananth Dhananjay

In [1]:
import re
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = "../data/AReM"
RNG_SEED = 552  # bootstrap seed

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a)

Note: bending2/dataset4 uses spaces instead of commas, and cycling/dataset9 and dataset14 have an extra comma at the end. The loader handles both.

In [2]:
COLS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES = COLS[1:]

def read_arem_file(path):
    rows = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            fields = [f for f in re.split(r"[,\s]+", line) if f != ""]
            rows.append([float(f) for f in fields])
    df = pd.DataFrame(rows, columns=COLS)
    return df

# number of files in each activity folder
FILE_COUNTS = {
    "bending1": 7, "bending2": 6, "cycling": 15, "lying": 15,
    "sitting": 15, "standing": 15, "walking": 15,
}

records = []
for act, n in FILE_COUNTS.items():
    for i in range(1, n + 1):
        path = f"{DATA_DIR}/{act}/dataset{i}.csv"
        records.append({"activity": act, "dataset": i, "data": read_arem_file(path)})

print(len(records))
records[0]["data"].head()

88


,time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
0,0.0,39.25,0.43,22.75,0.43,33.75,1.3
1,250.0,39.25,0.43,23.00,0.00,33.00,0.0
2,500.0,39.25,0.43,23.25,0.43,33.00,0.0
3,750.0,39.50,0.50,23.00,0.71,33.00,0.0
4,1000.0,39.50,0.50,24.00,0.00,33.00,0.0


### (b)

In [3]:
def is_test(activity, ds):
    if activity in ("bending1", "bending2"):
        return ds in (1, 2)
    return ds in (1, 2, 3)

for r in records:
    r["split"] = "test" if is_test(r["activity"], r["dataset"]) else "train"

activity = pd.Series([r["activity"] for r in records], name="activity")
split = pd.Series([r["split"] for r in records], name="split")
split_counts = pd.crosstab(activity, split, margins=True)
split_counts

split,test,train,All
activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


### (c)
#### i.
Time-domain features commonly used for time series classification:
- min, max, mean, median
- standard deviation, variance
- quartiles/percentiles, range, interquartile range
- skewness, kurtosis
- root mean square (RMS), energy
- autocorrelation
- number of peaks, zero crossings

Sources: tsfresh documentation (list of features), Lara & Labrador (2013)

#### ii.

In [4]:
def extract_features(series):
    x = np.asarray(series, dtype=float)
    return {
        "min": x.min(),
        "max": x.max(),
        "mean": x.mean(),
        "median": np.median(x),
        "std": x.std(ddof=1),
        "1st_quart": np.percentile(x, 25),
        "3rd_quart": np.percentile(x, 75),
    }

def build_feature_table(recs):
    rows = []
    for r in recs:
        row = {"activity": r["activity"], "dataset": r["dataset"], "split": r["split"]}
        for i, col in enumerate(SERIES, start=1):
            for k, v in extract_features(r["data"][col]).items():
                row[f"{k}{i}"] = v
        rows.append(row)
    df = pd.DataFrame(rows)
    df.index = pd.RangeIndex(1, len(df) + 1, name="Instance")
    return df

features = build_feature_table(records)
FEAT_COLS = [c for c in features.columns if c not in ("activity", "dataset", "split")]

In [5]:
features[FEAT_COLS].round(3)

,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.625,40.50,1.477,39.25,42.000,0.0,1.30,0.359,0.430,0.323,0.00,0.500,4.00,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.000,1.120,27.25,38.25,34.311,35.00,2.188,33.000,36.00,0.00,1.92,0.571,0.43,0.583,0.000,1.300
2,38.00,45.67,42.813,42.50,1.436,42.00,43.670,0.0,1.22,0.372,0.470,0.289,0.00,0.500,2.00,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.000,1.145,27.67,38.50,33.025,33.00,1.995,32.000,34.50,0.00,3.11,0.571,0.43,0.601,0.000,1.300
3,35.00,47.40,43.954,44.33,1.559,43.00,45.000,0.0,1.70,0.426,0.470,0.339,0.00,0.500,6.50,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.000,0.830,29.00,38.50,35.588,36.00,2.000,35.362,36.50,0.00,1.79,0.493,0.43,0.514,0.000,0.940
4,33.00,47.75,42.180,43.50,3.671,39.15,45.000,0.0,3.00,0.696,0.500,0.631,0.00,1.120,8.50,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.430,1.300,20.00,38.67,33.494,35.00,3.849,30.458,36.33,0.00,2.18,0.614,0.50,0.524,0.000,1.000
5,33.00,45.75,41.678,41.75,2.243,41.33,42.750,0.0,2.83,0.536,0.500,0.405,0.43,0.710,3.00,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.430,1.120,23.67,37.50,29.857,30.00,2.411,28.458,31.25,0.00,1.79,0.383,0.43,0.389,0.000,0.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.587,34.25,4.651,30.25,37.000,0.0,14.67,4.577,4.260,2.461,2.87,6.105,7.33,23.25,15.318,15.330,2.970,13.25,17.270,0.0,9.00,3.227,3.10,1.647,2.050,4.260,6.33,25.00,15.892,16.00,3.284,13.730,18.25,0.00,8.32,3.260,3.11,1.640,2.050,4.322
85,19.75,45.50,34.323,35.25,4.752,31.00,38.000,0.0,13.47,4.456,3.900,2.599,2.49,5.970,6.25,22.25,15.210,15.250,2.711,13.50,17.000,0.0,9.00,3.277,3.11,1.500,2.170,4.390,6.25,23.25,15.550,15.75,3.120,13.500,17.75,0.00,9.67,3.433,3.20,1.733,2.157,4.565
86,19.50,46.00,34.546,35.25,4.842,31.25,37.812,0.0,12.47,4.372,4.135,2.360,2.49,5.800,7.00,22.67,15.186,15.250,2.758,13.25,17.000,0.0,8.34,3.029,2.86,1.665,1.790,4.110,7.00,22.75,15.760,15.75,2.823,14.000,17.75,0.00,10.00,3.338,3.08,1.657,2.160,4.335


#### iii.
90% bootstrap CI for the standard deviation of each feature (percentile method, 10,000 resamples).

In [6]:
def boot_ci_std(x, n_resamples=10000, conf=0.90, seed=RNG_SEED):
    res = stats.bootstrap(
        (np.asarray(x),),
        lambda a, axis: np.std(a, ddof=1, axis=axis),
        n_resamples=n_resamples,
        confidence_level=conf,
        method="percentile",
        random_state=np.random.default_rng(seed),
    )
    return res.confidence_interval.low, res.confidence_interval.high

ci_rows = []
for col in FEAT_COLS:
    lo, hi = boot_ci_std(features[col].values)
    ci_rows.append({"feature": col, "std": features[col].std(ddof=1), "CI_90_low": lo, "CI_90_high": hi})
std_ci = pd.DataFrame(ci_rows).set_index("feature")
std_ci.round(3)

,std,CI_90_low,CI_90_high
feature,,,
min1,9.570,8.262,10.761
max1,4.394,3.328,5.285
mean1,5.336,4.704,5.886
median1,5.440,4.790,6.004
std1,1.772,1.572,1.950
1st_quart1,6.154,5.582,6.652
3rd_quart1,5.139,4.332,5.841
min2,0.000,0.000,0.000
max2,5.063,4.623,5.402


#### iv.
I picked min, mean, and max. Mean gives the overall level of the signal and min/max give its range, which should differ between the static activities (sitting, standing, lying, bending) and the moving ones (walking, cycling). Median and the quartiles are close to the mean, so they don't add much.

## 2. ISLR 3.7.4

(a) The cubic regression will have lower (or equal) training RSS. The cubic model contains the linear model, so least squares can only fit the training data as well or better with the extra terms.

(b) The linear regression should have lower test RSS. Since the true relationship is linear, the extra cubic terms just fit noise and overfit.

(c) The cubic regression will have lower (or equal) training RSS, for the same reason as (a). This doesn't depend on the true relationship.

(d) Not enough information. If the true relationship is close to linear, the linear model could have lower test RSS. If it is far from linear, the cubic model should have lower test RSS.